<a href="https://colab.research.google.com/github/KeerthanaSistla/Natural-Language-Processing/blob/main/NLP_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Machine Translation using LSTM (English → French)

In [20]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

input_texts = ["hello", "how are you", "good morning"]
target_texts = ["<start> bonjour <end>", "<start> comment ca va <end>", "<start> bonjour <end>"]

tokenizer_in = Tokenizer(filters='')
tokenizer_in.fit_on_texts(input_texts)
encoder_input = pad_sequences(tokenizer_in.texts_to_sequences(input_texts), padding='post')

tokenizer_out = Tokenizer(filters='')
tokenizer_out.fit_on_texts(target_texts)
decoder_input = pad_sequences(tokenizer_out.texts_to_sequences(target_texts), padding='post')

decoder_target = np.zeros_like(decoder_input)
for i in range(decoder_input.shape[0]):
    decoder_target[i, :-1] = decoder_input[i, 1:]
decoder_target = np.expand_dims(decoder_target, -1)

vocab_in = len(tokenizer_in.word_index) + 1
vocab_out = len(tokenizer_out.word_index) + 1

latent_dim = 128

encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(vocab_in, 64)(encoder_inputs)
_, state_h, state_c = LSTM(latent_dim, return_state=True)(enc_emb)

decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(vocab_out, 64)
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])

decoder_dense = Dense(vocab_out, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.fit([encoder_input, decoder_input], decoder_target, epochs=200, verbose=0)

encoder_model = Model(encoder_inputs, [state_h, state_c])

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, h, c = decoder_lstm(dec_emb2, initial_state=[decoder_state_input_h, decoder_state_input_c])
decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model([decoder_inputs, decoder_state_input_h, decoder_state_input_c],
                      [decoder_outputs2, h, c])

reverse_target_index = {i: w for w, i in tokenizer_out.word_index.items()}
reverse_target_index[0] = ""

start_token = tokenizer_out.word_index['<start>']
end_token = tokenizer_out.word_index['<end>']

def translate(sentence):
    seq = pad_sequences(tokenizer_in.texts_to_sequences([sentence]), maxlen=encoder_input.shape[1], padding='post')
    states = encoder_model.predict(seq, verbose=0)
    target_seq = np.array([[start_token]])
    result = []

    for _ in range(10):
        output, h, c = decoder_model.predict([target_seq] + states, verbose=0)
        idx = np.argmax(output[0, -1, :])
        word = reverse_target_index.get(idx, "")
        if word == "<end>" or word == "":
            break
        result.append(word)
        target_seq = np.array([[idx]])
        states = [h, c]
    return " ".join(result)

for s in input_texts:
    print(s, "->", translate(s))

hello -> bonjour
how are you -> comment ca va
good morning -> bonjour


#2. CNN Sentence Classification (IMDB dataset)

In [21]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 10000
max_len = 200

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

model = Sequential([
    Embedding(vocab_size, 128, input_length=max_len),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=3, batch_size=128, validation_data=(X_test, y_test))

loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/3


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


196/196 ━━━━━━━━━━━━━━━━━━━━ 88s 440ms/step - accuracy: 0.7341 - loss: 0.5139 - val_accuracy: 0.8639 - val_loss: 0.3157
Epoch 2/3
196/196 ━━━━━━━━━━━━━━━━━━━━ 79s 404ms/step - accuracy: 0.9006 - loss: 0.2564 - val_accuracy: 0.8800 - val_loss: 0.2823
Epoch 3/3
196/196 ━━━━━━━━━━━━━━━━━━━━ 90s 444ms/step - accuracy: 0.9598 - loss: 0.1271 - val_accuracy: 0.8870 - val_loss: 0.2779
782/782 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.8870 - loss: 0.2779
Test Accuracy: 0.8870000243186951


#3. Pretrained Translation

In [1]:
!pip install sentencepiece -q

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

sentences = [
    "Artificial intelligence is powerful",
    "Machine learning is fascinating",
    "I love programming"
]

inputs = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True)

outputs = model.generate(**inputs)

translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)

for i, t in enumerate(translations):
    print(f"English: {sentences[i]}")
    print(f"French : {t}")
    print()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

English: Artificial intelligence is powerful
French : L'intelligence artificielle est puissante

English: Machine learning is fascinating
French : L'apprentissage automatique est fascinant

English: I love programming
French : J'adore la programmation



#4. Question Answering

In [3]:
from transformers import pipeline

qa = pipeline("question-answering")

context = "Machine learning is a subset of artificial intelligence that learns from data."

res = qa(question="What is machine learning?", context=context)

print(res["answer"])
print(res["score"])

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

a subset of artificial intelligence
0.2709541618824005


#5. Text Generation

In [18]:
import random
from collections import defaultdict

text = """
AI is changing the world. AI is useful in healthcare. AI is used in education.
Machine learning is part of AI. Deep learning improves AI systems.
"""

words = text.lower().split()

bigrams = defaultdict(list)

for i in range(len(words) - 1):
    bigrams[words[i]].append(words[i + 1])

def generate(prompt, n=25):
    word = prompt.lower().split()[0]
    result = [word]

    for _ in range(n - 1):
        if word not in bigrams:
            break
        word = random.choice(bigrams[word])
        result.append(word)

    return " ".join(result)

print(generate("AI", 30))

ai is useful in education. machine learning is part of ai. deep learning improves ai is used in healthcare. ai is used in healthcare. ai systems.


#6. Text Summarization

In [25]:
!pip install nltk -q

import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from collections import defaultdict
import heapq

text = """
Artificial intelligence is transforming industries by enabling machines
to learn from data and improve over time. AI is widely used in healthcare,
finance, education, and transportation. Companies use AI to automate tasks,
improve efficiency, and make better decisions. AI systems can process large
amounts of data quickly and accurately.
"""

sentences = sent_tokenize(text)

stop_words = set(stopwords.words("english"))

word_freq = defaultdict(int)

words = word_tokenize(text.lower())

for word in words:
    if word.isalnum() and word not in stop_words:
        word_freq[word] += 1

sentence_scores = defaultdict(int)

for sentence in sentences:
    for word in word_tokenize(sentence.lower()):
        if word in word_freq:
            sentence_scores[sentence] += word_freq[word]

summary_sentences = heapq.nlargest(
    2,
    sentence_scores,
    key=sentence_scores.get
)

summary = " ".join(summary_sentences)

print("Original Text:\n")
print(text)

print("\nSummary:\n")
print(summary)

Original Text:


Artificial intelligence is transforming industries by enabling machines
to learn from data and improve over time. AI is widely used in healthcare,
finance, education, and transportation. Companies use AI to automate tasks,
improve efficiency, and make better decisions. AI systems can process large
amounts of data quickly and accurately.


Summary:

Companies use AI to automate tasks,
improve efficiency, and make better decisions. 
Artificial intelligence is transforming industries by enabling machines
to learn from data and improve over time.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


#7. Topic Modeling

In [26]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

documents = [
    "Artificial intelligence and machine learning are transforming technology.",
    "Deep learning is a subset of machine learning.",
    "Football and cricket are popular sports.",
    "Many people enjoy watching cricket matches.",
    "AI is used in healthcare and finance.",
    "Sports competitions bring excitement to fans."
]

vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(documents)

lda = LatentDirichletAllocation(n_components=2, random_state=42)
lda.fit(X)

words = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(lda.components_):
    print(f"\nTopic {topic_idx + 1}:")
    top_words = [words[i] for i in topic.argsort()[-5:]]
    print(" ".join(top_words))


Topic 1:
artificial technology intelligence machine learning

Topic 2:
healthcare used finance sports cricket


#8. Transformer Encoder (PyTorch)

In [27]:
import torch
import torch.nn as nn

class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super(TransformerEncoderBlock, self).__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_output, _ = self.attention(x, x, x)

        x = self.norm1(x + self.dropout(attn_output))

        ff_output = self.feed_forward(x)

        x = self.norm2(x + self.dropout(ff_output))

        return x

batch_size = 2
seq_length = 5
embed_dim = 16

x = torch.rand(batch_size, seq_length, embed_dim)

encoder = TransformerEncoderBlock(
    embed_dim=16,
    num_heads=2,
    ff_dim=32
)

output = encoder(x)

print("Input Shape :", x.shape)
print("Output Shape:", output.shape)

Input Shape : torch.Size([2, 5, 16])
Output Shape: torch.Size([2, 5, 16])


#Text Classification Model (Naive Bayes)

In [28]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Sample training data
texts = [
    "I love this movie",
    "This film was amazing",
    "I hate this movie",
    "This film was terrible",
    "What a great experience",
    "Very bad acting"
]

# Labels
labels = [
    "Positive",
    "Positive",
    "Negative",
    "Negative",
    "Positive",
    "Negative"
]

# Create pipeline
model = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('classifier', MultinomialNB())
])

# Train model
model.fit(texts, labels)

# Test sentences
test_sentences = [
    "I enjoyed the film",
    "The movie was bad"
]

# Predictions
predictions = model.predict(test_sentences)

# Print results
for text, prediction in zip(test_sentences, predictions):
    print(f"Text: {text}")
    print(f"Prediction: {prediction}\n")

Text: I enjoyed the film
Prediction: Negative

Text: The movie was bad
Prediction: Negative



#Text Classification (Logistic Regression)

In [29]:
# Import libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Sample dataset
texts = [
    "I love this movie",
    "This film was amazing",
    "What a great experience",
    "I really enjoyed the story",
    "This movie was fantastic",

    "I hate this movie",
    "This film was terrible",
    "Very bad acting",
    "I disliked the story",
    "Worst movie ever"
]

labels = [
    "Positive",
    "Positive",
    "Positive",
    "Positive",
    "Positive",

    "Negative",
    "Negative",
    "Negative",
    "Negative",
    "Negative"
]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42
)

# Create pipeline
model = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', LogisticRegression())
])

# Train model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

# Detailed report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# Test with new sentences
new_texts = [
    "The movie was awesome",
    "I did not like the film"
]

predictions = model.predict(new_texts)

# Display predictions
print("\nPredictions:\n")

for text, pred in zip(new_texts, predictions):
    print(f"Text: {text}")
    print(f"Prediction: {pred}\n")

Accuracy: 0.0

Classification Report:

              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00       1.0
    Positive       0.00      0.00      0.00       1.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0


Predictions:

Text: The movie was awesome
Prediction: Positive

Text: I did not like the film
Prediction: Negative

